#### ***Single AI Agent***

In [1]:
import os 
import requests
import warnings
warnings.filterwarnings("ignore")

from langchain_groq import ChatGroq 
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.tools import tool 
from langchain.agents import create_react_agent,AgentExecutor

In [2]:
from  dotenv import load_dotenv
load_dotenv

GROQ = os.getenv("GRQO_API_KEY")
TAVILY = os.getenv("TAVILY_API_KEY")
WHEATHER = os.getenv("WHEATHERSTACK_API_KEY")

In [3]:
search_tool = TavilySearchResults(max_results=3)

# Making Tool 
@tool
def get_wheather_data(city:str) -> str:
    """
    Fetch current weather information for a city.
    """
    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WHEATHER}&query={city}"
    )
    response = requests.get(url)
    data = response.json()
    if "current" not in data:
        return f"could not fetched current data for {city}"
    return(
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [4]:
result = search_tool.invoke("give me latest news on AI")
result

[{'url': 'https://www.artificialintelligence-news.com',
  'content': 'Telecoms Tech News\n\nJune 19, 2026\n\n## Spectrum AI expands commercial CBRS network capacity fivefold\n\nMarketing Tech News\n\nJune 19, 2026\n\n## Warner Bros Discovery expands agentic AI use in ad buying\n\n## Subscribe\n\nAll our premium content and latest tech news delivered straight to your inbox\n\n## Explore\n\n## Reach Our Audience\n\n## Categories\n\n## Other Publications\n\nAI News is part of TechForge\n\n## Subscribe\n\nAll our premium content and latest tech news delivered straight to your inbox [...] AI Business Strategy, AI in Action, AI Market Trends, Featured News, Features, Finance AI, Inside AI, Retail & Logistics AI, World of Work\n\n### Visa ChatGPT integration enables AI agent retail purchasing\n\nAI Business Strategy, AI in Action, AI Market Trends, Computer Vision, Featured News, Features, Finance AI, How It Works, Inside AI, Natural Language Processing (NLP), Trust, Bias & Fairness, World of

In [5]:
## LLM  

llm = ChatGroq( 
    model = "qwen/qwen3-32b", 
    temperature=0, 
    api_key=GROQ
)

In [6]:
response = llm.invoke("Tell me Joke about AI")
response

AIMessage(content='<think>\nOkay, the user wants a joke about AI. Let me think about how to approach this. First, I need to come up with something that\'s both funny and relevant. Maybe play on common AI themes like machine learning, algorithms, or everyday AI applications.\n\nHmm, maybe a pun or a play on words. Let me brainstorm some ideas. How about something involving a robot or an AI misunderstanding human things? Like a robot trying to make a joke but taking it too literally. Or maybe an AI that\'s overconfident in its abilities.\n\nWait, here\'s an idea: an AI that\'s been trained on jokes but doesn\'t get the humor. Maybe it tells a joke that\'s technically correct but misses the point. For example, a dad joke about machine learning. Let me structure it.\n\nSetup: Why did the AI fail at telling dad jokes? Because it tried to optimize the punchline for maximum laughter but overfit to the training data. Then the punchline could be that all its jokes were about neural networks and

In [7]:
## PROMPTS
prompt = hub.pull("hwchase17/react")

d:\AIML_TechStack\Agentic_AI\venv\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [8]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [9]:
import warnings
from pydantic.warnings import PydanticDeprecatedSince20

warnings.filterwarnings(
    "ignore",
    category=PydanticDeprecatedSince20
)

In [10]:
## TOOLS 
tools = [search_tool, get_wheather_data]

## CREATE AGENT  
agent = create_react_agent(
    llm = llm,
    tools = tools,
    prompt=prompt
)

## EXECUTOR 
agent_executor = AgentExecutor( 
    agent=agent, 
    tools=tools, 
    verbose=False, # seen step process 
    handle_parsing_errors=True
)
## RUN AGENT 
response = agent_executor.invoke({
    "input":("find capital of india and current weather")
})
print(response["output"])

The capital of India is New Delhi. The current weather in New Delhi is 34°C with a severe sandstorm and 35% humidity.
